In [1]:
from sqlalchemy import create_engine
import pandas as pd, requests as req

# List of 20 countries based on GDP, from Wikipedia

In [2]:
'''
    Getting the top 30 countries based on GDP (source : Wikipedia)
'''

def get_top_20_countries():
    from io import StringIO
    import re

    url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:154.0) Gecko/20100101 Firefox/154.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
    } # To emulate firefox headers
    
    
    response = req.get(url, headers=headers)
    
    tables = pd.read_html(StringIO(response.text))
    
    # Inspect the tables
    for i, table in enumerate(tables):
        if table.shape[0]>200:
            countries = table[1:31]["Country/Territory"] # The table includes one extra row as cumulative global GDP
            countries = [c.lower() for c in countries]
            countries = [re.findall(r"^[^\[]*", c)[0] for c in countries]
            return countries
    return None

top_economy_countries = pd.DataFrame(get_top_20_countries(),columns=['country'])

In [3]:
top_economy_countries["id"] = top_economy_countries.index+1

In [4]:
pd.DataFrame(top_economy_countries,columns=['country']).to_csv('./covid_data/cleaned/countries.csv')

# Cleaning and trimming the Covid data
- The source for the data: [OurWorlInData](https://ourworldindata.org/covid-deaths)
- Since the goal is to utilize SQL queries, we reatin only the necessary columns in each table.
- The list of tables:
    1. New cases/deaths on a daily basis
       - *Columns:* country, date, new_cases, new_deaths

    2. Excess mortality compare to average weekly deaths between 2015 to 2019
       - *Columns:* country, date, time, time_unit (weekly, monthly), average_deaths_2015_2019_all_ages, deaths_since_2020_all_ages

    3. New covid tests administared
       - *Columns:* country, date, new_tests
         
    5. Vaccination data
       - *Columns:* country, date, total_vaccinations, people_vaccinated, people_fully_vaccinated


- For each table, every NaN in integer columns are set to zero. There were no NaN in other columns.

In [5]:
data = pd.read_csv('./covid_data/original/cases_deaths.csv')

data["country"] = data["country"].str.lower()
data = data[data.country.isin(top_economy_countries["country"])].reset_index(drop=True)

data["date"] = pd.to_datetime(data["date"])

In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 69281 entries, 0 to 69280
Data columns (total 40 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   country                                  69281 non-null  str           
 1   date                                     69281 non-null  datetime64[us]
 2   new_cases                                64795 non-null  float64       
 3   total_cases                              69281 non-null  int64         
 4   new_deaths                               65947 non-null  float64       
 5   total_deaths                             69281 non-null  int64         
 6   weekly_cases                             64650 non-null  float64       
 7   weekly_deaths                            65802 non-null  float64       
 8   weekly_pct_growth_cases                  51018 non-null  float64       
 9   weekly_pct_growth_deaths                 42478 non

In [7]:
data = data[['country', 'date', 'new_cases', 'new_deaths']] ## Chosen columns
data[data.isna()] = 0 ## Deal with NaN

data = data.merge(         # Replacing country with their ID
    top_economy_countries,
    on="country",
    suffixes=("", "_country")
)  
data.rename(columns={'id':'country_id'},inplace=True)
data.drop(columns='country',inplace=True)

In [8]:
data.to_csv('./covid_data/cleaned/cases_deaths.csv')

In [9]:
data = pd.read_csv('./covid_data/original/excess_mortality.csv')

In [10]:
data = data.rename(columns = {'entity':'country'})
data["country"] = data["country"].str.lower()
data = data[data.country.isin(top_economy_countries["country"])].reset_index(drop=True)

data["date"] = pd.to_datetime(data["date"])

In [11]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4146 entries, 0 to 4145
Data columns (total 43 columns):
 #   Column                                        Non-Null Count  Dtype         
---  ------                                        --------------  -----         
 0   country                                       4146 non-null   str           
 1   date                                          4146 non-null   datetime64[us]
 2   time                                          4146 non-null   int64         
 3   time_unit                                     4146 non-null   str           
 4   average_deaths_2015_2019_all_ages             1050 non-null   float64       
 5   p_avg_0_14                                    2704 non-null   float64       
 6   p_avg_15_64                                   2704 non-null   float64       
 7   p_avg_65_74                                   2704 non-null   float64       
 8   p_avg_75_84                                   2704 non-null   float64       
 9

In [12]:
data = data[['country', 'date', 'time', 'time_unit',
       'average_deaths_2015_2019_all_ages', 'deaths_since_2020_all_ages']] ## Chosen columns
data[data.isna()] = 0 ## Deal with NaN

data = data.merge(         # Replacing country with their ID
    top_economy_countries,
    on="country",
    suffixes=("", "_country")
)  
data.rename(columns={'id':'country_id'},inplace=True)
data.drop(columns='country',inplace=True)

In [13]:
data.to_csv('./covid_data/cleaned/excess_mortality.csv')

In [14]:
data = pd.read_csv('./covid_data/original/testing.csv')

In [15]:
data["country"] = data["country"].str.lower()
data = data[data.country.isin(top_economy_countries["country"])].reset_index(drop=True)

data["date"] = pd.to_datetime(data["date"])

In [16]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 23413 entries, 0 to 23412
Data columns (total 8 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   country                               23413 non-null  str           
 1   date                                  23413 non-null  datetime64[us]
 2   total_tests                           20523 non-null  float64       
 3   new_tests                             20195 non-null  float64       
 4   total_tests_per_thousand              20523 non-null  float64       
 5   new_tests_per_thousand                20195 non-null  float64       
 6   new_tests_7day_smoothed               23204 non-null  float64       
 7   new_tests_per_thousand_7day_smoothed  23204 non-null  float64       
dtypes: datetime64[us](1), float64(6), str(1)
memory usage: 1.4 MB


In [17]:
data = data[['country', 'date', 'new_tests']] ## Chosen columns
data[data.isna()] = 0 ## Deal with NaN

data = data.merge(         # Replacing country with their ID
    top_economy_countries,
    on="country",
    suffixes=("", "_country")
)  
data.rename(columns={'id':'country_id'},inplace=True)
data.drop(columns='country',inplace=True)

In [18]:
data.to_csv('./covid_data/cleaned/testing.csv')

In [19]:
data = pd.read_csv('./covid_data/original/vaccinations_global.csv')

In [20]:
data["country"] = data["country"].str.lower()
data = data[data.country.isin(top_economy_countries["country"])].reset_index(drop=True)

data["date"] = pd.to_datetime(data["date"])

In [21]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 29322 entries, 0 to 29321
Data columns (total 29 columns):
 #   Column                                                   Non-Null Count  Dtype         
---  ------                                                   --------------  -----         
 0   country                                                  29322 non-null  str           
 1   date                                                     29322 non-null  datetime64[us]
 2   total_vaccinations                                       20921 non-null  float64       
 3   people_vaccinated                                        19088 non-null  float64       
 4   people_fully_vaccinated                                  18436 non-null  float64       
 5   total_boosters                                           17007 non-null  float64       
 6   daily_vaccinations                                       19534 non-null  float64       
 7   daily_vaccinations_smoothed                              292

In [22]:
data = data[['country', 'date', 'total_vaccinations', 'people_vaccinated',
       'people_fully_vaccinated']] ## Chosen columns
data[data.isna()] = 0 ## Deal with NaN

data = data.merge(         # Replacing country with their ID
    top_economy_countries,
    on="country",
    suffixes=("", "_country")
)  
data.rename(columns={'id':'country_id'},inplace=True)
data.drop(columns='country',inplace=True)

In [23]:
data.to_csv('./covid_data/cleaned/vaccinations_global.csv')

# Uploading the tables into the postgres DB

In [28]:
dict_data = {'countries':'./covid_data/cleaned/countries.csv',
            'covid_cases':'./covid_data/cleaned/cases_deaths.csv',
            'vaccinations':'./covid_data/cleaned/vaccinations_global.csv',
            'excess_mortatility':'./covid_data/cleaned/excess_mortality.csv',
            'testing':'./covid_data/cleaned/testing.csv'}


In [31]:
import getpass

engine = create_engine(f'postgresql://{getpass.getpass("User for the server: ")}:{getpass.getpass("Passwrod for postgres server:")}@localhost:5432/{input("DB name:")}')

try:
    connection = engine.connect()
    print("Connection to PostgreSQL established successfully!")
except Exception as e:
    print(f"Connection failed! Error: {e}")

User for the server:  ········
Passwrod for postgres server: ········
DB name: coviddb


Connection to PostgreSQL established successfully!


In [32]:
from sqlalchemy import Table, Column, Integer, TEXT, REAL, Date, ForeignKey
from sqlalchemy import MetaData, text 

types = {"int16":Integer,
         "int32":Integer,
         "int64":Integer,
         "float16":REAL,
         "float32":REAL,
         "float64":REAL,
         "str":TEXT,
         "datetime64[us]":Date
        }

## Function for checking existing tables, droping constraints if needed, and droping all rows

In [29]:
def table_check_exists(engine, table_name):
    with engine.connect() as conn:
        r = conn.execute(text(f'''
                select exists (
                    select 1
                    from information_schema.tables
                    where table_schema = 'public'
                          and table_name = '{table_name}'
                    );
        '''))
        
    return r.fetchall()[0][0]

def table_drop_constraint(engine, table_name, constraint):
    
    if table_check_exists(engine = engine, table_name = table_name):
        with engine.begin() as conn:
            try:
                conn.execute(text(f"""
                    alter table {table_name} drop constraint {table_name}_{constraint};
                    """))
            except Exception as e:
                print(e)

def table_delrows(engine, table_name):
    
    if table_check_exists(engine = engine, table_name = table_name):
        with engine.begin() as conn:
            try:
                conn.execute(text(f'''
                truncate table {table_name} restart identity cascade;
                '''))
            except Exception as e:
                print(e)

In [33]:
for name in dict_data.keys():
    if name !='countries':
        table_drop_constraint(engine = engine , table_name = name, constraint = 'country_id_fkey')
        table_delrows(engine = engine , table_name = name)

table_delrows(engine = engine , table_name = 'countries')

## Uploading the tables

In [44]:
metadata_obj = MetaData()

for table_name,path in dict_data.items():
    df = pd.read_csv(path)
    df.drop(columns=["Unnamed: 0"],inplace=True)
    df['id'] = df.index+1
    if 'date' in df.columns:
        df['date'] = df['date'].astype("datetime64[us]")

    column_list = [Column(col,
                           types[df[col].dtype.name],
                           primary_key=True if col=='id' else False,
                          nullable=False if col=='id' else True) for idx,col in enumerate(df.columns) if col!='country_id']
    if 'country_id' in df.columns:
        column_list += [Column('country_id',
                           types[df['country_id'].dtype.name],
                           ForeignKey("countries.id"),
                           primary_key=False,
                          nullable=False)]
    
    table = Table(
                    table_name,
                    metadata_obj,
                    *column_list
                    )
    
    try:
        table.drop(engine, checkfirst=True)
    except Exception as e:
        print(f'Warning: {e}')
    try:
        table.create(engine, checkfirst=True)
    except:
        pass

In [45]:
for table_name,path in dict_data.items():
    df = pd.read_csv(path)
    df.drop(columns=["Unnamed: 0"],inplace=True)
    df['id'] = df.index+1

    query = f''' INSERT INTO {table_name} ( {', '.join(df.columns[:-1])+' ,'+df.columns[-1]} )
                 VALUES ( :{', :'.join(df.columns[:-1])+', :'+df.columns[-1]} )
                 ON CONFLICT (id) DO NOTHING'''
    
    with engine.begin() as conn:
        conn.execute(text(query), df.to_dict(orient='records'))